In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 5b: air temperature, leave-one-block-out

Design and decision rule fixed with the user, before any air model was scored (`docs/plan.md`).

- **Target:** REMS `air_temp_k`. Rows without ground temperature (dropped by the loader) and rows with `air_missing` are not used.
- **Ground input:** `ground_pred` = fold baseline + fold `flann_safe_no_ti` residual, out of fold for every row. Held-out rows of block b use the fold-b ground model. Training rows (block c != b) use their own out-of-fold value, from the fold-c ground model. **Known leak:** the fold-c ground model saw block b's ground observations (never its air temperatures). Removing it would need nested LOBO with physics refits (about 72 fits); it was judged small and accepted.
- **Learner:** the FLANN ridge basis (`residual.make_flann_basis`) with `baseline_k` set to `ground_pred`. Ridge alpha 1.0 on standardized columns, fixed. No clamp, no elevation.

| Configuration | Number | Static terms | Role |
|---|---|---|---|
| `air_hybrid` | 10 | `ground_pred` x (1, sin, cos of daily harmonic 1) | candidate |
| `air_time_only` | 11 | none | attribution |

Both share daily harmonics 1-3 and seasonal harmonics 1-2, each x (1, tau), and tau.

**Benchmark:** mean REMS air temperature per (LTST bin, 30-deg Ls bin) from the fold's training rows; per-bin mean where a cell is empty.

**Decision rule:** adopt `air_hybrid` if its macro RMSE beats the benchmark by at least 0.2 K **and** it is better in at least 5 of 9 blocks. Otherwise publish the better-scoring of the two configurations anyway, with the shortfall stated. The benchmark is not a candidate.

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r

SUSPECT_SOLS = (3763, 3778)
SUSPECT_BLOCK = 10                 # block 10 is exactly sols 3763-3778 (asserted below)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
ADOPT_MIN_GAIN_K, ADOPT_MIN_BLOCK_WINS = 0.2, 5
CANDIDATE, ATTRIBUTION = "air_hybrid", "air_time_only"
REPRO_TOL_K = 0.05                 # recomputed ground_pred vs Step 5's flann_rows.csv
GROUND_RANGE_PCT = (1.0, 99.0)     # rover range of ground_pred, reported for the map
NIGHT_HOURS = [0, 1, 2, 3, 4, 5, 19, 20, 21, 22, 23]
TREE_YARDSTICK_K = (3.5, 3.7)      # brief's tree model for air: 7 blocks, older split, not comparable
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
FLANN_ROWS = paths.RESIDUAL_OUT_DIR / "flann_rows.csv"
OUT = paths.AIR_OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = OUT / "air_progress.log"
PROGRESS_LOG.write_text("")
print("ground model:", r.GROUND_VARIANT, r.FLANN_VARIANTS[r.GROUND_VARIANT])
print("air variants:", r.AIR_VARIANTS, "| ridge alpha", r.FLANN_RIDGE_ALPHA)


def progress(line):
    print(line)
    with open(PROGRESS_LOG, "a") as fh:
        fh.write(line + "\n")

## Data, fold baselines and the out-of-fold ground prediction

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"
assert set(tbl.loc[tbl.block == SUSPECT_BLOCK, "sol"]) == set(tbl.sol[tbl.sol.between(*SUSPECT_SOLS)]), \
    "block 10 is not exactly the suspect sols"
per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs_ground = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
tbl["q_abs_wm2"] = 0.0     # not in the FLANN basis; make_features needs the column

fits = pd.read_csv(FIT_PARAMS).set_index("fold")
fold_base = {}
t0 = time.time()
for label in fits.index:
    p = tuple(fits.loc[label, list(t.PARAM_NAMES)].to_numpy(dtype=float))
    fold_base[label] = evaluate(p).copy()
    assert np.all(np.isfinite(fold_base[label])), f"non-finite baseline in {label}"
progress(f"fold baselines in {time.time() - t0:.0f} s")

# out-of-fold ground prediction: fold-b baseline + fold-b ground residual model, on block b's rows
static_g, clamp_g = r.FLANN_VARIANTS[r.GROUND_VARIANT]
ground_pred = np.full(len(tbl), np.nan)
for b in blocks:
    label = f"block {b}"
    held = block == b
    frame = tbl.assign(baseline_k=fold_base[label])
    m = r.FlannModel(static_g, clamp_g, frame[fit_ok & ~held], (obs_ground - fold_base[label])[fit_ok & ~held])
    ground_pred[held] = fold_base[label][held] + m.predict(frame[held])[0]
assert np.all(np.isfinite(ground_pred)), "ground_pred incomplete"
step5 = pd.read_csv(FLANN_ROWS)
assert (step5.sol.to_numpy() == tbl.sol.to_numpy()).all() and (step5.bin.to_numpy() == tbl.bin.to_numpy()).all(), \
    "row order differs from Step 5"
diff = np.abs(ground_pred - step5[f"pred_{r.GROUND_VARIANT}"].to_numpy()).max()
progress(f"max |recomputed ground_pred - Step 5 pred_{r.GROUND_VARIANT}| = {diff:.2e} K")
assert diff < REPRO_TOL_K, "ground_pred does not reproduce Step 5"
tbl["ground_pred_k"] = ground_pred
gp_resid = ground_pred - obs_ground
print(f"ground_pred error on all loaded rows: RMSE {np.sqrt(np.mean(gp_resid ** 2)):.2f} K, bias {gp_resid.mean():+.2f} K")

n_before = len(tbl)
tbl = tbl[~tbl.air_missing].reset_index(drop=True)
assert np.all(np.isfinite(tbl.air_temp_k)), "air temperature missing but not flagged"
progress(f"dropped {n_before - len(tbl)} rows with air_missing; {len(tbl)} rows remain "
         f"(the loader had already dropped rows without ground temperature)")
block = tbl.block.to_numpy()
fit_ok = (tbl.group == "main").to_numpy()
obs = tbl.air_temp_k.to_numpy()
print(tbl.group.value_counts().to_string())

## LOBO: benchmark, `air_time_only`, `air_hybrid`

In [ ]:
frame = tbl.assign(baseline_k=tbl.ground_pred_k)   # the FLANN basis reads baseline_k; here it is ground_pred
models = {v: {} for v in r.AIR_VARIANTS}
for v in r.AIR_VARIANTS:
    tbl[f"pred_{v}"] = np.nan
tbl["bench_k"] = np.nan
n_fallback = {}
t0 = time.time()
for b in blocks + ["all"]:
    held = block == b if b != "all" else np.zeros(len(tbl), bool)
    train = fit_ok & ~held
    for v, static in r.AIR_VARIANTS.items():
        m = r.FlannModel(static, False, frame[train], obs[train])
        models[v][b] = m
        if held.any():
            tbl.loc[held, f"pred_{v}"] = m.predict(frame[held])[0]
    if held.any():
        bench, n_fallback[b] = r.benchmark(tbl[~held], tbl[held], target="air_temp_k")
        tbl.loc[held, "bench_k"] = bench
        progress(f"fold {b:2d}: {int(train.sum())} training rows | benchmark fallback rows {n_fallback[b]} | "
                 f"{time.time() - t0:.1f} s")
assert tbl[["bench_k"] + [f"pred_{v}" for v in r.AIR_VARIANTS]].notna().all().all()
n_cols = {v: models[v]["all"].basis(frame.head(1))[0].shape[1] for v in r.AIR_VARIANTS}
print("basis columns:", n_cols)
assert n_cols == {CANDIDATE: 24, ATTRIBUTION: 21}, "basis size differs from the plan"
print("benchmark fallback rows, total:", sum(n_fallback.values()))

## Scores and the decision rule

In [ ]:
MODELS = ["bench", ATTRIBUTION, CANDIDATE]
COLS = {"bench": "bench_k", **{m: f"pred_{m}" for m in r.AIR_VARIANTS}}
for m in MODELS:
    tbl[f"resid_{m}"] = tbl[COLS[m]] - tbl.air_temp_k


def score(g, models=MODELS):
    out = {"n": len(g)}
    for m in models:
        out[f"{m}_rmse_k"] = np.sqrt((g[f"resid_{m}"] ** 2).mean())
        out[f"{m}_bias_k"] = g[f"resid_{m}"].mean()
    return pd.Series(out)


scored = tbl[tbl.group != "placeholder_tau"]
main = tbl[tbl.group == "main"]
by_block = scored.groupby("block").apply(score, include_groups=False)
by_group = tbl.groupby("group").apply(score, include_groups=False)
summary = pd.DataFrame({
    "macro_rmse_k": {m: by_block[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_3763_k": {m: by_block.drop(index=SUSPECT_BLOCK)[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_block0_k": {m: by_block.drop(index=0)[f"{m}_rmse_k"].mean() for m in MODELS},
    "pooled_rmse_main_k": {m: np.sqrt((main[f"resid_{m}"] ** 2).mean()) for m in MODELS},
    "rmse_placeholder_tau_k": {m: by_group.loc["placeholder_tau", f"{m}_rmse_k"] for m in MODELS},
    "rmse_suspect_terrain_k": {m: by_group.loc["suspect_terrain", f"{m}_rmse_k"] for m in MODELS},
    "blocks_beating_bench": {m: int((by_block[f"{m}_rmse_k"] < by_block.bench_rmse_k).sum()) for m in MODELS},
})
pd.options.display.float_format = "{:.2f}".format
print(summary.to_string())
print(f"(reference only, not comparable: the brief's tree yardstick for air, {TREE_YARDSTICK_K[0]}-{TREE_YARDSTICK_K[1]} K macro)")
print()
print("Per-block RMSE (K):")
print(by_block[["n"] + [f"{m}_rmse_k" for m in MODELS]].to_string())
print()
print("Per-block bias (K):")
print(by_block[[f"{m}_bias_k" for m in MODELS]].to_string())

macro = summary.macro_rmse_k
gain = macro["bench"] - macro[CANDIDATE]
wins = summary.blocks_beating_bench[CANDIDATE]
adopted = gain >= ADOPT_MIN_GAIN_K and wins >= ADOPT_MIN_BLOCK_WINS
published = CANDIDATE if adopted else min((CANDIDATE, ATTRIBUTION), key=lambda m: macro[m])
log = [f"{CANDIDATE} vs benchmark: macro {macro[CANDIDATE]:.3f} K vs {macro['bench']:.3f} K, gain {gain:+.3f} K "
       f"(need >= {ADOPT_MIN_GAIN_K}), better in {wins}/9 blocks (need >= {ADOPT_MIN_BLOCK_WINS}) "
       f"-> {'adopted' if adopted else 'not adopted'}",
       f"{ATTRIBUTION}: macro {macro[ATTRIBUTION]:.3f} K, better than the benchmark in "
       f"{summary.blocks_beating_bench[ATTRIBUTION]}/9 blocks",
       f"published air model: {published}" + ("" if adopted else
       f" (fails the rule; shortfall against the benchmark {macro[published] - macro['bench']:+.3f} K macro, "
       f"to be stated in the manifest and README)")]
print()
print(chr(10).join(log))
(OUT / "air_decision.txt").write_text(chr(10).join(log) + chr(10))

## Errors by hour, season and block 0; how air responds to ground

In [ ]:
by_hour = main.groupby("ltst_hour").apply(score, include_groups=False)
by_ls = main.groupby("ls_bin").apply(score, include_groups=False)
print("By LTST hour (main rows):")
print(by_hour[[f"{m}_bias_k" for m in MODELS] + [f"{m}_rmse_k" for m in MODELS]].to_string())
print()
print("By Ls bin (main rows):")
print(by_ls[["n"] + [f"{m}_bias_k" for m in MODELS]].to_string())

b0 = tbl[tbl.block == 0]
night0, day0 = b0[b0.ltst_hour.isin(NIGHT_HOURS)], b0[~b0.ltst_hour.isin(NIGHT_HOURS)]
fold0 = pd.DataFrame({m: {"rmse_k": np.sqrt((b0[f"resid_{m}"] ** 2).mean()), "bias_k": b0[f"resid_{m}"].mean(),
                          "night_bias_k": night0[f"resid_{m}"].mean(), "day_bias_k": day0[f"resid_{m}"].mean()}
                      for m in MODELS}).T
g0 = b0.ground_pred_k - b0.ground_temp_k
print()
print(f"Block 0 (held out): ground_pred error RMSE {np.sqrt((g0 ** 2).mean()):.2f} K, bias {g0.mean():+.2f} K")
print(fold0.to_string())

# d(air)/d(ground_pred) by hour, all-blocks hybrid model: +1 K on every training row's ground_pred
hyb = models[CANDIDATE]["all"]
fr = frame[fit_ok]
d_air = hyb.predict(fr.assign(baseline_k=fr.baseline_k + 1.0))[0] - hyb.predict(fr)[0]
sens = pd.Series(d_air).groupby(fr.ltst_hour.to_numpy()).mean()
print()
print("all-blocks air_hybrid: change in air per +1 K of ground_pred, by LTST hour (K/K):")
print(sens.round(3).to_string())
lo_gp, hi_gp = np.percentile(fr.baseline_k, GROUND_RANGE_PCT)
print(f"rover range of ground_pred on fitting rows ({GROUND_RANGE_PCT[0]:g}-{GROUND_RANGE_PCT[1]:g} pct): "
      f"{lo_gp:.1f}-{hi_gp:.1f} K; min-max {fr.baseline_k.min():.1f}-{fr.baseline_k.max():.1f} K")

In [ ]:
tbl[["sol", "bin", "block", "group", "ltst_hour", "ls_bin", "air_temp_k", "ground_temp_k", "ground_pred_k",
     "bench_k"] + [f"pred_{v}" for v in r.AIR_VARIANTS]].to_csv(OUT / "air_rows.csv", index=False)
summary.to_csv(OUT / "air_summary.csv")
by_block.to_csv(OUT / "air_by_block.csv")
by_group.to_csv(OUT / "air_by_group.csv")
by_hour.to_csv(OUT / "air_by_ltst_hour.csv")
by_ls.to_csv(OUT / "air_by_ls_bin.csv")
fold0.to_csv(OUT / "air_fold0.csv")
sens.rename("dair_dground_k_per_k").to_csv(OUT / "air_ground_sensitivity_by_hour.csv")
pd.Series({"ground_pred_p01_k": lo_gp, "ground_pred_p99_k": hi_gp,
           "ground_pred_min_k": fr.baseline_k.min(), "ground_pred_max_k": fr.baseline_k.max()}).to_csv(
    OUT / "air_ground_pred_range.csv")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
ax = axes[0]
x = np.arange(len(by_block))
for i, m in enumerate(MODELS):
    ax.bar(x + (i - 1) * 0.27, by_block[f"{m}_rmse_k"], 0.27, label=m)
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="Air temperature, leave-one-block-out RMSE")
ax.legend(fontsize=8)
ax = axes[1]
for m, st in zip(MODELS, ["o-", "s-", "^-"]):
    ax.plot(by_hour.index, by_hour[f"{m}_bias_k"], st, ms=3, label=f"{m} bias")
    ax.plot(by_hour.index, by_hour[f"{m}_rmse_k"], st[0] + "--", ms=2, alpha=0.6, label=f"{m} RMSE")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="K", title="By LTST hour (main rows)")
ax.legend(fontsize=7)
ax = axes[2]
for i, m in enumerate(MODELS):
    ax.bar(by_ls.index + LS_BIN_DEG / 2 + (i - 1) * 8, by_ls[f"{m}_bias_k"], 8, label=m)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="Ls (deg)", ylabel="bias (K)", title="Bias by 30-degree Ls bin (main rows)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "air_scores.png", dpi=120)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey=True)
for ax, b in zip(axes.flat, blocks):
    g = tbl[tbl.block == b].groupby("ltst_hour")[["air_temp_k", "bench_k"] + [f"pred_{v}" for v in r.AIR_VARIANTS]].mean()
    ax.plot(g.index, g.air_temp_k, "k.-", label="REMS air")
    ax.plot(g.index, g.bench_k, "b.--", label="benchmark")
    ax.plot(g.index, g[f"pred_{ATTRIBUTION}"], "c.-", label=ATTRIBUTION)
    ax.plot(g.index, g[f"pred_{CANDIDATE}"], "m.-", label=CANDIDATE)
    s = tbl[tbl.block == b].sol
    ax.set_title(f"block {b}: sols {s.min()}-{s.max()} (held out)", fontsize=9)
axes.flat[0].legend(fontsize=7)
fig.supxlabel("LTST hour")
fig.supylabel("air temperature (K)")
fig.tight_layout()
fig.savefig(OUT / "air_diurnal_by_block.png", dpi=120)
print("saved to", OUT)